In [1]:
# 📦 Cell 0 - Install dependencies (run once)
# ------------------------------------------

# !pip install google-api-python-client tqdm pandas python-dotenv pyprojroot pyarrow fastparquet

In [2]:
# 📌 Cell 1 - 导入依赖库与路径初始化 (Industrial Version)
# -----------------------------------------------------------

import os
import glob
import time
import json
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime

from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from dotenv import load_dotenv

# 引入本地语种清洗 (防止非英文污染 NLP 模型)
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException
DetectorFactory.seed = 42

# 🌟 绝对定位项目根目录
from pyprojroot import here

PROJECT_ROOT = here()
DATA_DIR = PROJECT_ROOT / "Data"
ENV_FILE = PROJECT_ROOT / "env" / "youtube.env"
CONFIG_FILE = PROJECT_ROOT / "configs" / "config.yaml"

print(f"✅ Auto-detected Project Root: {PROJECT_ROOT}")

# 健壮的语言检测器
def is_english(text):
    if not text or len(str(text).strip()) < 5: return False
    try:
        return detect(text) == 'en'
    except LangDetectException:
        return False

✅ Auto-detected Project Root: d:\coding\projects\Python\Youtube Comment Sentiment Analysis


In [3]:
# 🔐 Cell 2 - 加载 API Key 与轮转调度引擎
# -----------------------------------------------------------

def load_all_youtube_api_keys():
    load_dotenv(dotenv_path=ENV_FILE) 
    keys, i = [], 1
    while True:
        key = os.getenv(f"YOUTUBE_API_KEY_{i}")
        if key:
            keys.append(key)
            i += 1
        else:
            break
    if not keys: raise ValueError("⚠️ No API keys found in .env file.")
    return keys

api_keys = load_all_youtube_api_keys()
print(f"✅ Loaded {len(api_keys)} API key(s).")

# 全局状态字典
state = {
    "client_index": 0,
    "youtube": build("youtube", "v3", developerKey=api_keys[0])
}

# 🌟 核心防弹包装器：自动轮转 + 指数退避
def execute_api_with_retry(request_creator, max_retries=5, base_delay=2):
    global state
    for attempt in range(max_retries):
        try:
            request = request_creator(state["youtube"])
            return request.execute()
        except HttpError as e:
            if e.resp.status in [403, 429]:
                if len(api_keys) > 1:
                    state["client_index"] = (state["client_index"] + 1) % len(api_keys)
                    print(f"\n  🔄 Quota Limit! Switching to API key #{state['client_index'] + 1}...")
                    state["youtube"] = build("youtube", "v3", developerKey=api_keys[state["client_index"]])
                    time.sleep(1)
                    continue
                time.sleep(base_delay * (2 ** attempt))
            elif e.resp.status == 404:
                return None 
            else:
                raise e
    print("❌ API 熔断！")
    return None

✅ Loaded 7 API key(s).


In [4]:
# 📥 Cell 3 - 自动对接管线：寻找 01 数据并清洗特征
# -----------------------------------------------------------

# 1. 自动寻找最新的 data_01 文件夹
search_pattern = str(DATA_DIR / "data_01_*")
data_01_dirs = sorted([d for d in glob.glob(search_pattern) if os.path.isdir(d)])

if not data_01_dirs:
    raise FileNotFoundError("❌ Cannot find any data_01 folders!")

latest_01_dir = data_01_dirs[-1]
input_csv = os.path.join(latest_01_dir, "01_video_candidates_final.csv")
print(f"🔍 Auto-detected latest input data: {input_csv}")

df_videos = pd.read_csv(input_csv)

# 🌟 确保契约字段存在 (brand, topic)
if 'brand' not in df_videos.columns:
    df_videos['brand'] = df_videos['cell_id'].apply(lambda x: 'SHEIN' if 'non_shein' not in x else 'NON_SHEIN')
if 'topic' not in df_videos.columns:
    df_videos['topic'] = df_videos['cell_id'].apply(lambda x: 'LABOR' if 'labor' in x else 'ENV')

# 2. 创建 02 输出专属目录
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = DATA_DIR / f"data_02_comments_replies_{timestamp}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\n📂 Outputs will be saved to: {OUTPUT_DIR}")

🔍 Auto-detected latest input data: d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_01_video_candidates_20260226_210856\01_video_candidates_final.csv

📂 Outputs will be saved to: d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_02_comments_replies_20260227_095101


In [5]:
# 🚜 Cell 4 - 引擎 A：抓取 Top-level Comments (扩容版)
# -----------------------------------------------------------
MAX_TOP_LEVEL = 3000 # 🌟 扩容：单视频主楼上限提高到 3000，为劳工组储备足够多的弹药

top_level_records = []

for _, row in tqdm(df_videos.iterrows(), total=len(df_videos), desc="Scraping Top-level"):
    vid, cell_id, brand, topic = row['video_id'], row['cell_id'], row['brand'], row['topic']
    page_token = None
    fetched_count = 0
    
    while fetched_count < MAX_TOP_LEVEL:
        def create_threads_req(client):
            return client.commentThreads().list(
                part="snippet", videoId=vid, order="relevance", maxResults=100, pageToken=page_token, textFormat="plainText"
            )
            
        res = execute_api_with_retry(create_threads_req)
        if not res or not res.get("items"): break
        
        for item in res["items"]:
            snippet = item["snippet"]["topLevelComment"]["snippet"]
            text_original = snippet.get("textOriginal", "")
            
            if not is_english(text_original): continue
                
            top_level_records.append({
                "comment_id": item["id"],
                "thread_id": item["id"],
                "video_id": vid,
                "cell_id": cell_id,
                "brand": brand,
                "topic": topic,
                "text_original": text_original,
                "published_at": snippet.get("publishedAt"),
                "like_count_parent": int(snippet.get("likeCount", 0)),
                "total_reply_count": int(item["snippet"].get("totalReplyCount", 0)),
                "source_api": "commentThreads.list"
            })
            fetched_count += 1
            if fetched_count >= MAX_TOP_LEVEL: break
            
        page_token = res.get("nextPageToken")
        if not page_token: break

df_top_level = pd.DataFrame(top_level_records)
df_top_level.to_parquet(OUTPUT_DIR / "top_level_comments.parquet", index=False)
print(f"✅ 顶层评论抓取完毕！共入库 {len(df_top_level)} 条纯净数据。")

Scraping Top-level:   0%|          | 0/74 [00:00<?, ?it/s]

✅ 顶层评论抓取完毕！共入库 26730 条纯净数据。


In [6]:
# 🧠 Cell 5 - 战术规划局：动态配平 Thread Index Plan (保证各组均衡)
# -----------------------------------------------------------
import math

# 🌟 核心配平策略：每个阵营（cell_id）目标需要多少个战场？
TARGET_THREADS_PER_CELL = 2000  
MIN_THREADS_PER_VIDEO = 50 # 即使视频再多，单个视频保底也要挖 50 个

print(f"🗺️ 正在生成战场勘探计划表 (目标：每组筹集约 {TARGET_THREADS_PER_CELL} 个线程)...")

df_potential = df_top_level[df_top_level['total_reply_count'] > 0].copy()
plan_list = []

# 按四大阵营分别进行动态计算
for cell_id, cell_group in df_potential.groupby('cell_id'):
    unique_videos = cell_group['video_id'].nunique()
    if unique_videos == 0: continue
    
    # 🌟 动态计算单视频额度 (不够的使劲挖，多的平摊)
    dynamic_top_n = max(MIN_THREADS_PER_VIDEO, math.ceil(TARGET_THREADS_PER_CELL / unique_videos))
    print(f"  📌 阵营 [{cell_id}]: {unique_videos} 个视频 -> 动态分配: 每个视频深挖 Top {dynamic_top_n} 线程")
    
    for vid, video_group in cell_group.groupby('video_id'):
        top_threads = video_group.sort_values(
            by=['total_reply_count', 'like_count_parent'], ascending=[False, False]
        ).head(dynamic_top_n).copy()
        
        top_threads['rank_in_video'] = range(1, len(top_threads) + 1)
        # 记录契约字段
        top_threads['dynamic_top_n_applied'] = dynamic_top_n
        plan_list.append(top_threads)

if not plan_list:
    raise ValueError("⚠️ 未找到任何带有回复的评论！")

df_thread_plan = pd.concat(plan_list, ignore_index=True)

df_thread_plan = df_thread_plan[[
    'thread_id', 'video_id', 'cell_id', 'brand', 'topic', 
    'total_reply_count', 'like_count_parent', 'rank_in_video', 'dynamic_top_n_applied'
]].rename(columns={'thread_id': 'parent_comment_id'})

df_thread_plan['thread_id'] = df_thread_plan['parent_comment_id']
df_thread_plan['reply_fetch_policy'] = "dynamic_topN_balance"
df_thread_plan['replies_fetched_count'] = 0
df_thread_plan['fetch_status'] = "planned"

df_thread_plan.to_parquet(OUTPUT_DIR / "thread_index.parquet", index=False)
print(f"🎯 作战计划锁定！动态配平完成，共准备深钻 {len(df_thread_plan)} 个核心战场。")

🗺️ 正在生成战场勘探计划表 (目标：每组筹集约 2000 个线程)...
  📌 阵营 [non_shein_environment]: 26 个视频 -> 动态分配: 每个视频深挖 Top 77 线程
  📌 阵营 [non_shein_labor]: 10 个视频 -> 动态分配: 每个视频深挖 Top 200 线程
  📌 阵营 [shein_environment]: 25 个视频 -> 动态分配: 每个视频深挖 Top 80 线程
  📌 阵营 [shein_labor]: 12 个视频 -> 动态分配: 每个视频深挖 Top 167 线程
🎯 作战计划锁定！动态配平完成，共准备深钻 3845 个核心战场。


In [7]:
# 🚁 Cell 6 - 引擎 B：连环战场深度钻探 (深挖版)
# -----------------------------------------------------------
# 🌟 扩容：提高回复抓取深度，为 07 模块的 HDBSCAN 聚类提供足够密集的对线文本
MAX_REPLIES_PER_THREAD = 500 

replies_records = []

for idx, row in tqdm(df_thread_plan.iterrows(), total=len(df_thread_plan), desc="Scraping Replies"):
    parent_id = row['parent_comment_id']
    vid, cell_id, brand, topic = row['video_id'], row['cell_id'], row['brand'], row['topic']
    
    page_token = None
    fetched_count = 0
    
    while fetched_count < MAX_REPLIES_PER_THREAD:
        def create_replies_req(client):
            return client.comments().list(
                part="snippet", parentId=parent_id, maxResults=100, pageToken=page_token, textFormat="plainText"
            )
            
        res = execute_api_with_retry(create_replies_req)
        if not res or not res.get("items"): break
        
        for item in res["items"]:
            snippet = item["snippet"]
            text_original = snippet.get("textOriginal", "")
            if not is_english(text_original): continue
                
            replies_records.append({
                "comment_id": item["id"],
                "thread_id": parent_id,
                "parent_comment_id": parent_id,
                "video_id": vid,
                "cell_id": cell_id,
                "brand": brand,
                "topic": topic,
                "text_original": text_original,
                "published_at": snippet.get("publishedAt"),
                "like_count_reply": int(snippet.get("likeCount", 0)),
                "source_api": "comments.list"
            })
            fetched_count += 1
            if fetched_count >= MAX_REPLIES_PER_THREAD: break
            
        page_token = res.get("nextPageToken")
        if not page_token: break
        
    df_thread_plan.at[idx, 'replies_fetched_count'] = fetched_count
    df_thread_plan.at[idx, 'fetch_status'] = "ok"

df_replies = pd.DataFrame(replies_records)
df_replies.to_parquet(OUTPUT_DIR / "replies.parquet", index=False)

# 更新计划表的状态
df_thread_plan.to_parquet(OUTPUT_DIR / "thread_index.parquet", index=False)

print(f"✅ 引擎 B 执行完毕！深度深潜斩获 {len(df_replies)} 条真实战场回复。")

Scraping Replies:   0%|          | 0/3845 [00:00<?, ?it/s]


  🔄 Quota Limit! Switching to API key #2...

  🔄 Quota Limit! Switching to API key #3...
✅ 引擎 B 执行完毕！深度深潜斩获 21936 条真实战场回复。


In [8]:
# 📊 Cell 7 - 融合面板构建：生成 thread_panel_base (为 05 模块打基础)
# -----------------------------------------------------------
import numpy as np

print("🏗️ 正在构建 05 模块多变量结构面板 (thread_panel_base)...")

# 1. 聚合回复数据 (计算楼中楼的总赞数)
reply_agg = df_replies.groupby('thread_id').agg(
    like_reply_sum=('like_count_reply', 'sum'),
    like_reply_mean=('like_count_reply', 'mean')
).reset_index()

# 2. 与主楼数据合并
df_panel = df_thread_plan.merge(
    df_top_level[['comment_id', 'text_original']].rename(columns={'comment_id': 'thread_id', 'text_original': 'parent_text_original'}),
    on='thread_id', how='left'
).merge(reply_agg, on='thread_id', how='left')

# 填补空值 (没有成功抓到英文回复的 thread)
df_panel['like_reply_sum'] = df_panel['like_reply_sum'].fillna(0).astype(int)
df_panel['like_reply_mean'] = df_panel['like_reply_mean'].fillna(0)

# 3. 提前计算 05 模块要求的 Engagement Index [cite: 8]
# engagement = log(1+replies) + 0.2*log(1+likes_parent) + 0.2*log(1+likes_reply_sum) 
df_panel['engagement_index'] = (
    np.log1p(df_panel['replies_fetched_count']) + 
    0.2 * np.log1p(df_panel['like_count_parent']) + 
    0.2 * np.log1p(df_panel['like_reply_sum'])
)

df_panel.to_parquet(OUTPUT_DIR / "thread_panel_base.parquet", index=False)

print(f"✅ 面板构建完成！(涵盖 {len(df_panel)} 个深度线程)")
display(df_panel[['thread_id', 'cell_id', 'total_reply_count', 'like_count_parent', 'like_reply_sum', 'engagement_index']].head())

# =================================================================
# 最终数据审计报告
# =================================================================
manifest = {
    "execution_time": timestamp,
    "top_level_comments_fetched": len(df_top_level),
    "high_controversy_threads_explored": len(df_thread_plan),
    "replies_fetched": len(df_replies),
    "schemas_delivered": ["top_level_comments.parquet", "thread_index.parquet", "replies.parquet", "thread_panel_base.parquet"]
}

with open(OUTPUT_DIR / "scrape_manifest.json", 'w') as f:
    json.dump(manifest, f, indent=4)
    
print("\n🎉 [MODULE 02 SUCCESS] 数据契约全部履约！可以直接进入 03 清洗 或 04 情绪判定！")

🏗️ 正在构建 05 模块多变量结构面板 (thread_panel_base)...
✅ 面板构建完成！(涵盖 3845 个深度线程)


,thread_id,cell_id,total_reply_count,like_count_parent,like_reply_sum,engagement_index
0,UgxWcLR0B5dPDcSTcAd4AaABAg,non_shein_environment,38,2020,467,6.362881
1,UgxaQKT6m0k-2LetXLZ4AaABAg,non_shein_environment,21,766,310,5.567498
2,UgzC-P1kh4ir5NWr-n54AaABAg,non_shein_environment,19,1974,130,5.488436
3,UgzdqkFdd6Mtrigs5yt4AaABAg,non_shein_environment,17,1663,56,5.064595
4,Ugw7QRXg9eyZXPa-5f94AaABAg,non_shein_environment,9,772,74,4.496139



🎉 [MODULE 02 SUCCESS] 数据契约全部履约！可以直接进入 03 清洗 或 04 情绪判定！


In [9]:
# 📊 Cell 8 - 资产战果分布盘点 (Data Asset Audit)
# -----------------------------------------------------------
print("\n" + "="*50)
print("📊 各阵营 (Cell) 战果分布大盘点")
print("="*50)

# 1. 统计主楼 (Top-level) 数量
top_level_stats = df_top_level.groupby('cell_id').size().reset_index(name='top_level_count')

# 2. 统计深挖的 Thread (战场) 数量
thread_stats = df_thread_plan.groupby('cell_id').size().reset_index(name='explored_threads_count')

# 3. 统计实际抓到的 Reply (楼中楼) 数量
reply_stats = df_replies.groupby('cell_id').size().reset_index(name='fetched_replies_count')

# 4. 合并报表
df_audit = top_level_stats.merge(thread_stats, on='cell_id', how='left').merge(reply_stats, on='cell_id', how='left')

# 填补空值并转为整数
df_audit = df_audit.fillna(0)
for col in ['explored_threads_count', 'fetched_replies_count']:
    df_audit[col] = df_audit[col].astype(int)

# 5. 计算总弹药量 (该阵营真正可用于 04 和 07 模块的纯英文文本总数)
df_audit['total_pure_texts'] = df_audit['top_level_count'] + df_audit['fetched_replies_count']

display(df_audit)

# 打印大盘总结
print(f"\n💡 全局战果总结：")
print(f"  - 🎯 总计储备主楼：{df_audit['top_level_count'].sum():,} 条")
print(f"  - ⚔️ 总计深挖战场：{df_audit['explored_threads_count'].sum():,} 个")
print(f"  - 💬 总计对线回复：{df_audit['fetched_replies_count'].sum():,} 条")
print(f"  - 🚀 最终可用总纯净数据量：{df_audit['total_pure_texts'].sum():,} 条")


📊 各阵营 (Cell) 战果分布大盘点


,cell_id,top_level_count,explored_threads_count,fetched_replies_count,total_pure_texts
0,non_shein_environment,10807,1267,9572,20379
1,non_shein_labor,3636,578,2942,6578
2,shein_environment,7515,1197,5761,13276
3,shein_labor,4772,803,3661,8433



💡 全局战果总结：
  - 🎯 总计储备主楼：26,730 条
  - ⚔️ 总计深挖战场：3,845 个
  - 💬 总计对线回复：21,936 条
  - 🚀 最终可用总纯净数据量：48,666 条
